# 15 — Extreme user discovery (MovieLens full scan)

Scan raw `rating.csv` + `movie.csv` for users with extreme contrarian behavior or radical taste shifts.  
Goal: find real users (beyond baseline User 666) to validate split-complex RBM channel divergence.

**Parts in this notebook**

| Part | Section |
|------|--------|
| Part 0 | Setup (paths, thresholds) |
| Part 1 | Helper functions |
| Part 2 | Strategy 1 — Ultimate Contrarians |
| Part 3 | Strategy 2 — Temporal Taste-Shifters |

> Streams `rating.csv` in chunks (~2 min on MovieLens 20M). Same logic as `scripts/find_extreme_users.py`.

In [1]:
# Part 0 — Setup (paths, thresholds)
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

rating_path = root / "data" / "rating.csv"
movie_path = root / "data" / "movie.csv"

assert rating_path.exists(), f"Missing {rating_path}"
assert movie_path.exists(), f"Missing {movie_path}"

CHUNK_SIZE = 1_000_000
CSV_DTYPES = {"userId": "int32", "movieId": "int32", "rating": "float32"}

# Strategy 1
MASTERPIECE_MEAN = 4.3
MASTERPIECE_MIN_COUNT = 100
DISASTER_MEAN = 2.0
DISASTER_MIN_COUNT = 100
LOW_ON_MASTERPIECE = 1.5
HIGH_ON_DISASTER = 4.5
MIN_CONTRARIAN_MOVIES_EACH = 3

# Strategy 2
HEAVY_USER_MIN_RATINGS = 200
GENRE_SWING_MIN = 2.5
MIN_GENRE_RATINGS_PER_ERA = 3
TOP_TASTE_SHIFTERS = 5

print(f"Project root: {root}")
print(f"Rating file:  {rating_path}")
print(f"Movie file:   {movie_path}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Rating file:  /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/rating.csv
Movie file:   /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/movie.csv


## Part 1 — Helper functions

In [2]:
# Part 1 — Helper functions

def load_movie_genres() -> pd.DataFrame:
    movies = pd.read_csv(movie_path, usecols=["movieId", "title", "genres"])
    movies["genres"] = movies["genres"].fillna("(no genres listed)")
    return movies


def compute_global_movie_stats() -> pd.DataFrame:
    sums: dict[int, float] = defaultdict(float)
    counts: dict[int, int] = defaultdict(int)
    print(f"Streaming {rating_path.name} for per-movie stats …")
    for chunk in pd.read_csv(
        rating_path,
        usecols=["movieId", "rating"],
        dtype=CSV_DTYPES,
        chunksize=CHUNK_SIZE,
    ):
        grp = chunk.groupby("movieId")["rating"].agg(["sum", "count"])
        for mid, row in grp.iterrows():
            mid = int(mid)
            sums[mid] += float(row["sum"])
            counts[mid] += int(row["count"])
    return pd.DataFrame(
        {
            "movieId": list(counts.keys()),
            "rating_count": [counts[m] for m in counts],
            "mean_rating": [sums[m] / counts[m] for m in counts],
        }
    )


def count_user_ratings() -> pd.Series:
    counts: dict[int, int] = defaultdict(int)
    print(f"Streaming {rating_path.name} for per-user counts …")
    for chunk in pd.read_csv(
        rating_path, usecols=["userId"], dtype=CSV_DTYPES, chunksize=CHUNK_SIZE
    ):
        vc = chunk["userId"].value_counts()
        for uid, c in vc.items():
            counts[int(uid)] += int(c)
    return pd.Series(counts, name="rating_count").sort_index()


def load_heavy_user_ratings(heavy_ids: set[int]) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    print(f"Loading ratings for {len(heavy_ids):,} heavy users …")
    for chunk in pd.read_csv(
        rating_path,
        usecols=["userId", "movieId", "rating", "timestamp"],
        dtype=CSV_DTYPES,
        chunksize=CHUNK_SIZE,
    ):
        sub = chunk[chunk["userId"].isin(heavy_ids)]
        if not sub.empty:
            parts.append(sub)
    ratings = pd.concat(parts, ignore_index=True)
    ratings["timestamp"] = pd.to_datetime(ratings["timestamp"])
    return ratings


def era_label_per_user(df: pd.DataFrame) -> np.ndarray:
    order = df.groupby("userId").cumcount()
    era_sizes = df.groupby("userId")["userId"].transform("count")
    return np.where(order < (era_sizes // 2), "early", "late")


def explode_genre_ratings(ratings: pd.DataFrame, movies: pd.DataFrame) -> pd.DataFrame:
    cols = ["userId", "movieId", "rating", "timestamp"]
    if "era" in ratings.columns:
        cols.append("era")
    merged = ratings[cols].merge(movies[["movieId", "genres"]], on="movieId", how="left")
    merged["genres"] = merged["genres"].fillna("(no genres listed)")
    return merged.assign(genre=merged["genres"].str.split("|")).explode("genre")


def find_taste_shifters(ratings: pd.DataFrame, movies: pd.DataFrame) -> pd.DataFrame:
    ratings = ratings.sort_values(["userId", "timestamp", "movieId"]).copy()
    ratings["era"] = era_label_per_user(ratings)
    genre_ratings = explode_genre_ratings(ratings, movies)
    era_means = (
        genre_ratings.groupby(["userId", "genre", "era"])["rating"]
        .agg(["mean", "count"])
        .reset_index()
    )
    era_means = era_means[era_means["count"] >= MIN_GENRE_RATINGS_PER_ERA]
    pivot_mean = era_means.pivot_table(
        index=["userId", "genre"], columns="era", values="mean"
    )
    if "early" not in pivot_mean.columns or "late" not in pivot_mean.columns:
        return pd.DataFrame()
    pivot_mean = pivot_mean.dropna(subset=["early", "late"])
    pivot_mean["delta"] = pivot_mean["late"] - pivot_mean["early"]
    hits: list[dict] = []
    for uid in pivot_mean.index.get_level_values("userId").unique():
        sub = pivot_mean.xs(uid, level="userId")
        if isinstance(sub, pd.Series):
            continue
        dropped = sub[sub["delta"] <= -GENRE_SWING_MIN]
        rose = sub[sub["delta"] >= GENRE_SWING_MIN]
        if dropped.empty or rose.empty:
            continue
        best_pair, best_score = None, -np.inf
        for g_drop, row_a in dropped.iterrows():
            for g_rise, row_b in rose.iterrows():
                if g_drop == g_rise:
                    continue
                score = (-row_a["delta"]) + row_b["delta"]
                if score > best_score:
                    best_score = score
                    best_pair = {
                        "userId": int(uid),
                        "genre_drop": g_drop,
                        "genre_rise": g_rise,
                        "early_mean_drop_genre": float(row_a["early"]),
                        "late_mean_drop_genre": float(row_a["late"]),
                        "drop_delta": float(row_a["delta"]),
                        "early_mean_rise_genre": float(row_b["early"]),
                        "late_mean_rise_genre": float(row_b["late"]),
                        "rise_delta": float(row_b["delta"]),
                        "swing_score": float(score),
                    }
        if best_pair:
            hits.append(best_pair)
    if not hits:
        return pd.DataFrame()
    return (
        pd.DataFrame(hits)
        .sort_values("swing_score", ascending=False)
        .drop_duplicates(subset=["userId"])
        .head(TOP_TASTE_SHIFTERS)
    )

## Part 2 — Strategy 1: Ultimate Contrarians

Users who rate **Global Masterpieces** (mean > 4.3, n > 100) very low (≤ 1.5) **and**  
rate **Global Disasters** (mean < 2.0, n > 100) very high (≥ 4.5), at least 3 movies each.

In [3]:
# Part 2 — Strategy 1: Ultimate Contrarians
movies = load_movie_genres()
movie_stats = compute_global_movie_stats()

masterpiece_set = set(
    movie_stats.loc[
        (movie_stats["mean_rating"] > MASTERPIECE_MEAN)
        & (movie_stats["rating_count"] > MASTERPIECE_MIN_COUNT),
        "movieId",
    ].astype(int)
)
disaster_set = set(
    movie_stats.loc[
        (movie_stats["mean_rating"] < DISASTER_MEAN)
        & (movie_stats["rating_count"] > DISASTER_MIN_COUNT),
        "movieId",
    ].astype(int)
)

print(f"Global Masterpieces: {len(masterpiece_set):,} movies")
print(f"Global Disasters:    {len(disaster_set):,} movies")
if masterpiece_set:
    display(
        movie_stats[movie_stats["movieId"].isin(masterpiece_set)]
        .merge(movies, on="movieId")
        .sort_values("mean_rating", ascending=False)
    )

low_rows, high_rows = [], []
print(f"\nScanning contrarian ratings …")
for chunk in pd.read_csv(
    rating_path,
    usecols=["userId", "movieId", "rating"],
    dtype=CSV_DTYPES,
    chunksize=CHUNK_SIZE,
):
    mp = chunk[chunk["movieId"].isin(masterpiece_set) & (chunk["rating"] <= LOW_ON_MASTERPIECE)]
    ds = chunk[chunk["movieId"].isin(disaster_set) & (chunk["rating"] >= HIGH_ON_DISASTER)]
    if not mp.empty:
        low_rows.append(mp)
    if not ds.empty:
        high_rows.append(ds)

low_df = pd.concat(low_rows, ignore_index=True) if low_rows else pd.DataFrame()
high_df = pd.concat(high_rows, ignore_index=True) if high_rows else pd.DataFrame()

mp_users = low_df.groupby("userId").size() if not low_df.empty else pd.Series(dtype=int)
ds_users = high_df.groupby("userId").size() if not high_df.empty else pd.Series(dtype=int)
qualifying = mp_users[mp_users >= MIN_CONTRARIAN_MOVIES_EACH].index.intersection(
    ds_users[ds_users >= MIN_CONTRARIAN_MOVIES_EACH].index
)

if len(qualifying) == 0:
    print("\n=== No users matched Strategy 1 ===")
    print(
        f"(Only {len(masterpiece_set)} Global Masterpieces with mean>{MASTERPIECE_MEAN} "
        f"and n>{MASTERPIECE_MIN_COUNT}; consider relaxing thresholds.)"
    )
    contrarian_profile = pd.DataFrame()
else:
    qual_set = set(int(u) for u in qualifying)
    contrarian_profile = pd.concat(
        [
            low_df[low_df["userId"].isin(qual_set)].assign(pool="masterpiece_low"),
            high_df[high_df["userId"].isin(qual_set)].assign(pool="disaster_high"),
        ],
        ignore_index=True,
    )
    contrarian_profile = contrarian_profile.merge(movies, on="movieId", how="left")
    contrarian_profile = contrarian_profile.merge(
        movie_stats[["movieId", "mean_rating", "rating_count"]].rename(
            columns={"mean_rating": "global_mean", "rating_count": "global_n"}
        ),
        on="movieId",
        how="left",
    ).sort_values(["userId", "pool", "rating"])
    print(f"\n=== Strategy 1: {len(qual_set)} Ultimate Contrarian(s) ===")
    display(contrarian_profile)

Streaming rating.csv for per-movie stats …
Global Masterpieces: 4 movies
Global Disasters:    173 movies


,movieId,rating_count,mean_rating,title,genres
1,318,63366,4.446990,"Shawshank Redemption, The (1994)",Crime|Drama
3,858,41355,4.364732,"Godfather, The (1972)",Crime|Drama
0,50,47006,4.334372,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
2,527,50054,4.310175,Schindler's List (1993),Drama|War



Scanning contrarian ratings …

=== No users matched Strategy 1 ===
(Only 4 Global Masterpieces with mean>4.3 and n>100; consider relaxing thresholds.)
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/extreme_users_strategy1.csv


## Part 3 — Strategy 2: Temporal Taste-Shifters

Heavy users (> 200 ratings). Split history into **Early** / **Late** era (equal count).  
Find users where one genre drops ≥ 2.5 pts and another rises ≥ 2.5 pts.

In [4]:
# Part 3 — Strategy 2: Temporal Taste-Shifters
user_counts = count_user_ratings()
heavy_ids = set(
    user_counts[user_counts > HEAVY_USER_MIN_RATINGS].index.astype(int).tolist()
)
print(f"Heavy users (>{HEAVY_USER_MIN_RATINGS} ratings): {len(heavy_ids):,}")

heavy_ratings = load_heavy_user_ratings(heavy_ids)
taste_shifters = find_taste_shifters(heavy_ratings, movies)

print(f"\n=== Strategy 2: Top {TOP_TASTE_SHIFTERS} Temporal Taste-Shifters ===")
if taste_shifters.empty:
    print("No users matched genre inversion criteria.")
else:
    taste_shifters = taste_shifters.assign(
        total_ratings=taste_shifters["userId"].map(user_counts)
    )
    display(taste_shifters)
    for _, row in taste_shifters.iterrows():
        print(
            f"\nuserId={int(row['userId'])}  (total_ratings={int(row['total_ratings'])})")
        print(
            f"  DROP  {row['genre_drop']:18s}  "
            f"early={row['early_mean_drop_genre']:.2f} → late={row['late_mean_drop_genre']:.2f}  "
            f"Δ={row['drop_delta']:+.2f}"
        )
        print(
            f"  RISE  {row['genre_rise']:18s}  "
            f"early={row['early_mean_rise_genre']:.2f} → late={row['late_mean_rise_genre']:.2f}  "
            f"Δ={row['rise_delta']:+.2f}"
        )

Streaming rating.csv for per-user counts …
Heavy users (>200 ratings): 26,599
Loading ratings for 26,599 heavy users …

=== Strategy 2: Top 5 Temporal Taste-Shifters ===


,userId,genre_drop,genre_rise,early_mean_drop_genre,late_mean_drop_genre,drop_delta,early_mean_rise_genre,late_mean_rise_genre,rise_delta,swing_score,total_ratings
0,33634,Mystery,War,4.5,2.0,-2.5,1.357143,3.875,2.517857,5.017857,233



userId=33634  (total_ratings=233)
  DROP  Mystery             early=4.50 → late=2.00  Δ=-2.50
  RISE  War                 early=1.36 → late=3.88  Δ=+2.52

Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/extreme_users_strategy2.csv
